In [1]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT / "src"))
ART = ROOT / "artifacts"
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
from plotting import series_styles, new_figure

plt.rcParams.update({"figure.dpi": 110, "font.size": 9})

def es(v, d=4):
    """Numero con formato de Colombia: punto de miles y coma decimal."""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return ""
    if isinstance(v, (int, np.integer)):
        return f"{int(v):,}".replace(",", ".")
    s = f"{float(v):,.{d}f}"
    return s.replace(",", "X").replace(".", ",").replace("X", ".")

def tabla(df, d=4, enteros=()):
    out = df.copy()
    for c in out.columns:
        if c in enteros:
            out[c] = out[c].map(lambda v: es(int(v)) if pd.notna(v) else "")
        elif pd.api.types.is_float_dtype(out[c]):
            out[c] = out[c].map(lambda v: es(v, d))
        elif pd.api.types.is_integer_dtype(out[c]):
            out[c] = out[c].map(es)
    display(out.style.hide(axis="index"))

def figura(ruta, ancho=None):
    display(Image(filename=str(ruta), width=ancho))

# 11. Anexo de reproducibilidad

Este anexo reúne lo necesario para reproducir el proyecto de punta a punta, como pide el enunciado: el entorno con versiones fijadas, el hardware, las semillas, la configuración de Spark, el manifiesto de los datos, el orden de ejecución y el registro de las desviaciones frente al plan y al enunciado.

## 11.1 Entorno

El entorno se crea con `conda env create -f environment.yml`, que fija la versión de Python y de cada paquete, y el kernel del libro se registra con `python -m ipykernel install --user --name dl_tarea1`. PySpark necesita Java 17; se usó Eclipse Temurin 17.0.20.1. `src/config.py` respeta la variable `JAVA_HOME` si está definida y, si no, busca `~/Java/jdk-17`. En Windows, `config.py` además fija el intérprete de los *workers* de PySpark, el host local `127.0.0.1` y un `HADOOP_HOME` vacío dentro de `artifacts/`, porque Hadoop exige esa variable aunque no se use `winutils.exe`. No se descargó ningún binario nativo de terceros. Las versiones con que se ejecutó este libro son las siguientes:

In [2]:
import platform
from importlib.metadata import version
paquetes = ["pyspark", "scikit-learn", "pandas", "numpy", "polars", "duckdb", "pyarrow", "kaggle", "lime", "shap", "optuna",
            "matplotlib", "scipy", "statsmodels", "jupyter-book", "myst-nb", "sphinxcontrib-bibtex", "ipykernel", "joblib", "psutil"]
v = pd.DataFrame({"paquete": ["python"] + paquetes, "versión": [platform.python_version()] + [version(p) for p in paquetes]})
tabla(v)

paquete,versión
python,3.11.16
pyspark,3.5.3
scikit-learn,1.5.2
pandas,2.2.3
numpy,1.26.4
polars,1.12.0
duckdb,1.1.3
pyarrow,17.0.0
kaggle,2.2.5
lime,0.2.0.1


## 11.2 Hardware

| Componente | Valor |
|---|---|
| Procesador | Intel Core i5-1235U: 2 núcleos de rendimiento con 2 hilos cada uno y 8 núcleos de eficiencia, 12 hilos, límite de potencia de 15 W |
| Memoria | 16 GB de RAM |
| Sistema operativo | Windows 11 |
| GPU | No se usó; todo el proyecto corre en CPU |
| Spark | Modo local (`local[*]` y `local[p]` en el benchmarking); sin clúster |

El enunciado asume entre 32 y 64 GB de RAM para trabajar con Avazu; con 16 GB, los tamaños de muestra y las grillas se redujeron de forma declarada antes de entrenar (capítulos 4 y 5).

## 11.3 Semillas, partición y decisiones fijadas a priori

| Elemento | Valor |
|---|---|
| Semillas | 13, 29, 47, 71 y 97 (`SEEDS` en `src/config.py`) |
| Partición temporal | train días 21 a 28, validación día 29, prueba día 30 |
| Métrica primaria | ROC AUC; nivel de significancia $\alpha = 0{,}05$ |
| Margen de equivalencia | $\delta = 0{,}01$ de AUC, declarado antes de ver los resultados (`artifacts/stats/decisiones_a_priori.json`) |
| Optuna | muestreador TPE con semilla 13 |
| Muestreo con DuckDB | `USING SAMPLE ... (reservoir, semilla)`; con un solo hilo es determinista |

Los scripts de interpretabilidad, ablación, benchmarking por etapas y punto de cruce fijan DuckDB en un hilo, de modo que sus muestras se reproducen exactamente. Los scripts del experimento principal (`sklearn_pipeline.py` y `spark_pipeline.py`) y las pruebas de escalamiento de `benchmarking.py` muestrearon con 8 hilos; con esa configuración, el muestreo por reservorio de DuckDB no es determinista, así que al volver a correrlos se obtienen muestras distintas del mismo tamaño y resultados equivalentes, pero no idénticos dígito a dígito.

## 11.4 Configuración de Spark

| Parámetro | Experimento principal | Benchmarking |
|---|---|---|
| `master` | `local[*]` | `local[1]`, `local[2]`, `local[4]`, `local[8]`, `local[12]` y `local[*]` |
| `spark.driver.memory` | 12g | 8g |
| `spark.memory.fraction` | 0,8 | 0,8 (por etapas: valor por defecto) |
| `spark.memory.storageFraction` | 0,6 | 0,6 (por etapas: valor por defecto) |
| `spark.sql.shuffle.partitions` | 12 | 12 |
| `spark.driver.host` y `spark.driver.bindAddress` | 127.0.0.1 | 127.0.0.1 |
| Spark UI | deshabilitada | habilitada solo en `stage_benchmark.py`, puerto 4050 |

En modo local no hay ejecutores separados: el driver y el ejecutor comparten la misma JVM, cuya memoria es `spark.driver.memory`, y el número de hilos de `local[p]` hace las veces de núcleos de ejecutor. Los datos de Spark se leen desde archivos Parquet planos escritos con pyarrow, porque leer el directorio particionado exige las librerías nativas de Hadoop en Windows (sección 5.1).

## 11.5 Manifiesto de datos

Los datos vienen de la competencia *Avazu CTR Prediction* de Kaggle {cite}`avazu2014`, que se descarga con la API de Kaggle; el token se guarda en `~/.kaggle/access_token` y nunca dentro del repositorio. `src/data_prep.py` convierte el CSV a Parquet particionado por día, y `src/manifiesto_datos.py` registra el número de filas, el tamaño, la fecha de modificación y el hash SHA-256 de cada partición. El hash del día 30 permite verificar que el conjunto de prueba es el mismo que se usó en todo el proyecto.

In [3]:
m = json.loads((ART / "eda" / "manifiesto_datos.json").read_text(encoding="utf-8"))
t = pd.DataFrame(m["particiones"])[["dia", "rol", "archivo", "filas", "bytes", "modificado", "sha256"]]
t["MB"] = t["bytes"] / 1024**2
tabla(t.drop(columns=["bytes"])[["dia", "rol", "archivo", "filas", "MB", "modificado", "sha256"]], d=1, enteros=("dia", "filas"))
print(f"Total: {es(m['total_filas'])} filas")

dia,rol,archivo,filas,MB,modificado,sha256
21,train,processed/day=21/data_0.parquet,4.122.995,"171,2",2026-11-05T21:37:32,10eb965c18549174ee46d388f9b9b0aba544ef08911c969f60ebfb5ed48bf0a0
22,train,processed/day=22/data_0.parquet,5.337.126,"216,2",2026-11-05T21:37:32,17cb8523ddb53fe00a230bb1c447f7e9298132bc18b2d655c2cee7bcc04f3098
23,train,processed/day=23/data_0.parquet,3.870.752,"160,3",2026-11-05T21:37:32,1b8d2580d070058c352263642a60d598d6844642273b191c618116a05a5560b4
24,train,processed/day=24/data_0.parquet,3.335.302,"136,0",2026-11-05T21:37:32,f9df5f9c7eda6ef93f6ab5b4c045b45d5f53aea669d73cb90651a56f117632e2
25,train,processed/day=25/data_0.parquet,3.363.122,"139,0",2026-11-05T21:37:32,29d2868cd7d0a19a611a4287780ffe34a0ed66457135b1ee582ee1fc1d7d69d9
26,train,processed/day=26/data_0.parquet,3.835.892,"159,2",2026-11-05T21:37:32,38edbb6e857c4d3ee1ccfa4a19ed4782625b44bf819d671ffa9bd49a86f80bad
27,train,processed/day=27/data_0.parquet,3.225.010,"136,9",2026-11-05T21:37:32,7f3fb8faf281677360dad9a9a88cbd3e43e7d5bf97e68545886c2747b3f67a32
28,train,processed/day=28/data_0.parquet,5.287.222,"217,8",2026-11-05T21:37:32,86ae1d0e008aab3293710d8dbcd7a7e3c2583a2eea85771ab037eddeb5b355a4
29,validacion,processed/day=29/data_0.parquet,3.832.608,"160,1",2026-11-05T21:37:32,0eaaa8893c60dfd6a05d681b1ee92b8953d80d70fa20751dc5baff2c1840d3ea
30,prueba,processed/day=30/data_0.parquet,4.218.938,"170,8",2026-11-05T21:37:32,359b9068b917123ed898533d9b04759494c32961c4cb9a4ebbbfb0edeffa5bb7


Total: 40.428.967 filas


## 11.6 Orden de ejecución

Desde la carpeta `src/`, con el entorno `dl_tarea1` activo:

| Paso | Script | Salida |
|---|---|---|
| 1 | `data_prep.py` | descarga Avazu y escribe `data/processed/day=*/` |
| 2 | `manifiesto_datos.py` | `artifacts/eda/manifiesto_datos.json` |
| 3 | `eda.py` | `artifacts/eda/` |
| 4 | `sklearn_pipeline.py` | `artifacts/sklearn/` (45 corridas) |
| 5 | `spark_pipeline.py` | `artifacts/spark/` (15 corridas) |
| 6 | `stats_tests.py` | `artifacts/stats/` |
| 7 | `benchmarking.py` | escalamiento, Amdahl y punto de cruce en `artifacts/benchmarking/` |
| 8 | `stage_benchmark.py` | tiempos por etapa y Spark UI en `artifacts/benchmarking/`; debe correr con la máquina sin otra carga |
| 9 | `interpretability.py`, luego con `--sensibilidad` y con `--discretizacion` | `artifacts/interpretability/` |
| 10 | `ablation.py` | `artifacts/ablation/` |
| 11 | `jupyter-book build .`, desde la raíz | `_build/html/` |

Los scripts de modelado guardan cada corrida al terminarla y, si se interrumpen, retoman desde la última corrida completa. Cada script deja su registro con marcas de tiempo en el `run_log` de su carpeta de `artifacts/`; según esos registros, la ablación tardó unas 2 horas, el benchmarking de escalamiento 1 hora y 40 minutos y la interpretabilidad con su sensibilidad unos 12 minutos. Los cuadernos del libro solo leen `artifacts/` y no vuelven a entrenar.

## 11.7 Registro de desviaciones

| Desviación | Motivo | Efecto y dónde se trata |
|---|---|---|
| Avazu en lugar de Criteo 1TB | sin clúster ni almacenamiento para 1 TB; el enunciado admite Avazu como dataset secundario | capítulo 1 |
| Muestras de scikit-learn de 80.085, 800.847 y 1.601.694 filas, no de 100 mil, 1 y 2 millones | `USING SAMPLE` de DuckDB se aplica antes del filtro por día | tamaños efectivos documentados en la sección 3.6 |
| MLP de Spark con hashing de 2^9 y 300.000 filas, GBT con solo 5 variables y 1 millón de filas | densificación de la entrada y acumulación de memoria con 16 GB | sección 5.3 |
| Grilla del MLP de Spark reducida a una combinación del espacio mandatado | el tiempo por combinación hacía inviable la grilla en el plazo | sección 5.2 |
| Columnas C1, C14 a C21 y la hora no usadas en el modelo principal | omisión detectada al redactar | ablación `con_C_y_hora`, sección 9.3: diferencia equivalente a cero dentro de $\delta$ |
| AUC del MLP de Spark recalculado desde las probabilidades | el evaluador de Spark usa `rawPrediction` y 1.000 intervalos | sección 5.4 |
| Interpretabilidad y ablación con muestreo de un hilo | el muestreo con varios hilos no era reproducible | sección 11.3 |
| Dos variantes corregidas de ponderación de clases | el sobremuestreo antes de la partición interna de parada temprana contaminaba la validación, y la parada por accuracy no detecta el sobreajuste con clases balanceadas | sección 9.3; se reportan las tres variantes |
| Benchmarking por etapas repetido | la primera corrida coincidió con otro proceso en la máquina | se reporta solo la corrida limpia |
| Punto de cruce medido de nuevo | la primera versión comparaba tiempos de scikit-learn de otra sesión y otra muestra, y Spark recibía los datos con `createDataFrame` desde pandas; con el mismo número de filas, su ajuste tardaba 40,0 s frente a 14,2 s del benchmarking por etapas | sección 7.4: ambos entornos en la misma sesión, con la misma muestra y entrada por Parquet |
| Hash de la prueba calculado al cierre | no se registró al crear la partición | sección 11.5 |
| Red con embeddings y TorchDistributor no desarrollada | extensión opcional del enunciado; plazo | sección 10.4 |
| Costo en la nube con tarifa de terceros | la página oficial de precios no se consultó | sección 7.6 |